# UNSW-NB15 Multi-Class Classification (V2 — Improved Pipeline)

Improvements over V1:
1. Balanced class weights for all models
2. Macro F1 added to evaluation metrics
3. Per-class classification report for best model
4. LightGBM and HistGradientBoosting added as models

In [ ]:
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from lightgbm import LGBMClassifier
import xgboost as xgb
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

import warnings
warnings.filterwarnings('ignore')

## 1. Data Loading & Preprocessing

In [ ]:
train_path = r'../Datasets/USNW/UNSW_NB15_training-set.parquet'
test_path = r'../Datasets/USNW/UNSW_NB15_testing-set.parquet'

train_df = pd.read_parquet(train_path)
test_df = pd.read_parquet(test_path)

# Drop binary 'label' and 'id' as we focus on 'attack_cat'
for col in ['id', 'label']:
    if col in train_df.columns: train_df = train_df.drop(col, axis=1)
    if col in test_df.columns: test_df = test_df.drop(col, axis=1)

# Handle missing values in attack_cat
train_df['attack_cat'] = train_df['attack_cat'].fillna('Normal').astype(str).str.strip()
test_df['attack_cat'] = test_df['attack_cat'].fillna('Normal').astype(str).str.strip()

X_train = train_df.drop('attack_cat', axis=1)
y_train = train_df['attack_cat']
X_test = test_df.drop('attack_cat', axis=1)
y_test = test_df['attack_cat']

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("\nTarget Distribution:")
print(y_train.value_counts())

In [ ]:
# Encode categorical features
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
for col in categorical_cols:
    le = LabelEncoder()
    unique_vals = list(set(X_train[col].astype(str).unique()) | set(X_test[col].astype(str).unique()))
    le.fit(unique_vals)
    X_train[col] = le.transform(X_train[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

# Scale with RobustScaler
numerical_cols = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()
scaler = RobustScaler()
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

# Encode target variable
target_encoder = LabelEncoder()
target_encoder.fit(list(set(y_train.unique()) | set(y_test.unique())))
y_train_enc = target_encoder.transform(y_train)
y_test_enc = target_encoder.transform(y_test)

print("\nCategorical columns encoded:", categorical_cols)
print("Preprocessing complete.")

## 2. Model Training & Comparison (with Balanced Weights)

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, class_weight='balanced', random_state=42
    ),
    "Random Forest (Balanced)": RandomForestClassifier(
        n_estimators=100, class_weight='balanced',
        random_state=42, n_jobs=-1
    ),
    "LightGBM (Balanced)": LGBMClassifier(
        n_estimators=200, class_weight='balanced',
        random_state=42, n_jobs=-1, verbose=-1
    ),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=200, class_weight='balanced', random_state=42
    ),
}

results = []
for name, model in models.items():
    print(f"Training {name}...")
    start = time.time()
    model.fit(X_train, y_train_enc)
    elapsed = time.time() - start
    
    preds = model.predict(X_test)
    acc = accuracy_score(y_test_enc, preds)
    prec = precision_score(y_test_enc, preds, average='weighted', zero_division=0)
    rec = recall_score(y_test_enc, preds, average='weighted', zero_division=0)
    w_f1 = f1_score(y_test_enc, preds, average='weighted', zero_division=0)
    m_f1 = f1_score(y_test_enc, preds, average='macro', zero_division=0)
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Weighted Precision': prec,
        'Weighted Recall': rec,
        'Weighted F1': w_f1,
        'Macro F1': m_f1,
        'Time (s)': elapsed
    })
    print(f"{name} - Acc: {acc:.4f}, Weighted F1: {w_f1:.4f}, Macro F1: {m_f1:.4f} ({elapsed:.1f}s)")

results_df = pd.DataFrame(results)
display(results_df)

## 3. Per-Class Performance (Best Model)

In [ ]:
# Find best model by Macro F1
best_idx = results_df['Macro F1'].idxmax()
best_model_name = results_df.loc[best_idx, 'Model']
print(f"Best model by Macro F1: {best_model_name}")
print(f"Macro F1: {results_df.loc[best_idx, 'Macro F1']:.4f}")
print()

best_model = list(models.values())[best_idx]
preds = best_model.predict(X_test)
print(classification_report(y_test_enc, preds, target_names=target_encoder.classes_, zero_division=0))

## 4. Visualizing Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Weighted F1 vs Macro F1
melted_f1 = results_df.melt(id_vars='Model', value_vars=['Weighted F1', 'Macro F1'],
                            var_name='Metric', value_name='Score')
sns.barplot(data=melted_f1, x='Model', y='Score', hue='Metric', ax=axes[0])
axes[0].set_title('Weighted F1 vs Macro F1 Comparison')
axes[0].set_ylim(0, 1.05)
axes[0].tick_params(axis='x', rotation=20)
axes[0].legend(loc='lower right')

# Plot 2: All metrics
melted_all = results_df.melt(id_vars='Model',
                             value_vars=['Accuracy', 'Weighted Precision', 'Weighted Recall', 'Weighted F1', 'Macro F1'],
                             var_name='Metric', value_name='Score')
sns.barplot(data=melted_all, x='Model', y='Score', hue='Metric', ax=axes[1])
axes[1].set_title('Full Metric Comparison')
axes[1].set_ylim(0, 1.05)
axes[1].tick_params(axis='x', rotation=20)
axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)

plt.tight_layout()
plt.show()

## 5. Save Results

In [ ]:
import os
import joblib

os.makedirs('../Models', exist_ok=True)
results_df.to_csv('../Models/unsw_v2_results.csv', index=False)
joblib.dump(scaler, '../Models/unsw_v2_scaler.pkl')
joblib.dump(target_encoder, '../Models/unsw_v2_label_encoder.pkl')

print("Saved: results, scaler, label_encoder")